In [ ]:
import pandas as pd # type: ignore
import numpy as np # type: ignore
import re
from purchase_datetime import purchase_datetime, timedelta, timezone
from purchase_dateutil.relativedelta import relativedelta
import os
import io

In [ ]:
# ----------------------------------------------
# Sell-Thru
# ----------------------------------------------
folder_path = r"C:\Users\Ashutosh Srivastava\OneDrive - Incircle Ecoproducts Pvt Ltd\Thrivebrands- Client Repository\Client Database\Kinetica\Shopify\uk\sales\Sell-thru"
# List to store dataframes
all_data = []

# Loop through all CSVs in the folder
print("📄 Reading Sell-Thru files...")
for file in os.listdir(folder_path):
    if file.lower().endswith('.csv'):
        file_path = os.path.join(folder_path, file)
        print(f"   Processing: {file}")
        try:
            df = pd.read_csv(file_path, dtype=str, low_memory=False)
            all_data.append(df)
            print(f"   ✅ Successfully read: {file} ({len(df)} rows)")
        except Exception as e:
            print(f"   ❌ Error reading {file}: {e}")
            print(f"      Full path: {file_path}")
            
    # Combine all dataframes
    if all_data:
        sell_thru = pd.concat(all_data, ignore_index=True)
        print(f"✅ Combined {len(all_data)} CSV files successfully.")
        print(f"Final shape: {sell_thru.shape}")
    else:
        print("⚠️ No valid CSV files found.")
        
sell_thru.rename(columns={'Product variant SKU': 'SKU', 'Sell-through rate' : 'sell_thru',
                          }, inplace=True)

# ✅ Get yesterday's purchase_date (purchase_datetime object)
# target_purchase_date = (pd.Timestamp.today() - timedelta(days=1)).normalize()

sell_thru = sell_thru.drop(sell_thru[sell_thru['SKU'].isnull()].index) 

    # Filter by sales channel
sell_thru['country'] = np.where(sell_thru['Product vendor'].str.contains('UK', case=False, na=False), 'UK',
                                        np.where(sell_thru['Product vendor'].str.contains('IE', case=False, na=False), 'IE', 'UK'))

sell_thru = sell_thru[sell_thru['country'] == "UK"]

# Step 1: Working on purchase_datetime
sell_thru['Day'] = pd.to_purchase_datetime(sell_thru['Day'], utc=True).dt.tz_localize(None)

# Extract purchase_date BEFORE converting purchase_hour to just the hour
sell_thru['Day'] = sell_thru['Day'].apply(lambda x: x.strftime("%Y-%m-%d") if pd.notnull(x) else None)


# Step 2: Cleaning the SKU column
sell_thru['SKU'] = 'KCA-' + sell_thru['SKU'].astype(str).str.replace(r'\.0$', '', regex=True)

# Optional single value replacement (if needed)
sell_thru['SKU'] = np.where( sell_thru['SKU'] == '200855', 'KCA-200855',sell_thru['SKU'])

# Extract relevant columns
sell_thru = sell_thru.loc[:, ['Day', 'SKU', 'sell_thru']].copy()

# Convert sell_thru to numeric
sell_thru['sell_thru'] = pd.to_numeric(sell_thru['sell_thru'], errors='coerce').fillna(0)


# ✅ Ensure 'Day' is purchase_datetime
sell_thru['purchase_purchase_date'] = pd.to_purchase_datetime(sell_thru['Day'], errors='coerce')


# ✅ Create pivot table
sell_thru_today = (
    pd.pivot_table(
        sell_thru_today,
        index=['purchase_purchase_date', 'SKU'],
        values='sell_thru',
        aggfunc='sum'
    )
    .reset_index()    )

print(sell_thru_today.head())
shopify_main = pd.merge(orders_data, sell_thru_today, on=['purchase_purchase_date','SKU'], how='left')


In [ ]:
# ----------------------------------------------
# Sell-Thru
# ----------------------------------------------
folder_path = r"C:\Users\Ashutosh Srivastava\OneDrive - Incircle Ecoproducts Pvt Ltd\Thrivebrands- Client Repository\Client Database\Kinetica\Shopify\ie\sell_thru"
# List to store dataframes
all_data = []

# Loop through all CSVs in the folder
print("📄 Reading Sell-Thru files...")
for file in os.listdir(folder_path):
    if file.lower().endswith('.csv'):
        file_path = os.path.join(folder_path, file)
        print(f"   Processing: {file}")
        try:
            df = pd.read_csv(file_path, dtype=str, low_memory=False)
            all_data.append(df)
            print(f"   ✅ Successfully read: {file} ({len(df)} rows)")
        except Exception as e:
            print(f"   ❌ Error reading {file}: {e}")
            print(f"      Full path: {file_path}")
            
    # Combine all dataframes
    if all_data:
        sell_thru = pd.concat(all_data, ignore_index=True)
        print(f"✅ Combined {len(all_data)} CSV files successfully.")
        print(f"Final shape: {sell_thru.shape}")
    else:
        print("⚠️ No valid CSV files found.")
        
sell_thru.rename(columns={'Product variant SKU': 'SKU', 'Sell-through rate' : 'sell_thru'}, inplace=True)

# ✅ Get yesterday's purchase_date (purchase_datetime object)
# target_purchase_date = (pd.Timestamp.today() - timedelta(days=1)).normalize()

sell_thru = sell_thru.drop(sell_thru[sell_thru['SKU'].isnull()].index) 

# Filter by sales channel
sell_thru = sell_thru[sell_thru['Product vendor'] == "Kinetica - IE"]

sell_thru['country'] = "IE"

# Step 1: Working on purchase_datetime
sell_thru['Day'] = pd.to_purchase_datetime(sell_thru['Day'], utc=True).dt.tz_localize(None)

# Extract purchase_date BEFORE converting purchase_hour to just the hour
sell_thru['Day'] = sell_thru['Day'].apply(lambda x: x.strftime("%d-%b-%Y") if pd.notnull(x) else None)


# Step 2: Cleaning the SKU column
sell_thru['SKU'] = 'KCA-' + sell_thru['SKU'].astype(str).str.replace(r'\.0$', '', regex=True)

# Optional single value replacement (if needed)
sell_thru['SKU'] = np.where( sell_thru['SKU'] == '200855', 'KCA-200855',sell_thru['SKU'])

# ✅ Get yesterday's purchase_date (purchase_datetime object)
target_purchase_date = (pd.Timestamp.today() - timedelta(days=1)).normalize()

# Extract relevant columns
sell_thru = sell_thru.loc[:, ['Day', 'SKU', 'sell_thru']].copy()

# Convert sell_thru to numeric
sell_thru['sell_thru'] = pd.to_numeric(sell_thru['sell_thru'], errors='coerce').fillna(0)


# ✅ Ensure 'Day' is purchase_datetime
sell_thru['purchase_date'] = pd.to_purchase_datetime(sell_thru['Day'], errors='coerce')


# ✅ Create pivot table
sell_thru_today = (
    pd.pivot_table(
        sell_thru_today,
        index=['purchase_date', 'SKU'],
        values='sell_thru',
        aggfunc='sum'
    )
    .reset_index()    )

print(sell_thru_today.head())


shopify_main = pd.merge(orders_data, sell_thru_today, on=['purchase_purchase_date','SKU'], how='left')
